In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------------
# 基础模块 (CBR, Head, DownSample)
# --------------------------

class CBR(nn.Sequential):
    def __init__(self, ins, outs, k, s, p, d=1):
        super(CBR, self).__init__(
            nn.Conv2d(in_channels=ins, out_channels=outs, kernel_size=k, stride=s, padding=p, dilation=d, bias=False),
            nn.BatchNorm2d(num_features=outs),
            nn.ReLU(inplace=True)
        )

class DownSample(nn.Module):
    def __init__(self, ins, outs, k, s):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=ins, out_channels=outs, kernel_size=k, stride=s, bias=False),
            nn.BatchNorm2d(num_features=outs)
        )
    def forward(self, x):
        return self.block(x)

class Head(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.1),
            nn.Conv2d(512, num_classes, kernel_size=1, stride=1) # 最终分类投影
        )
    def forward(self, x):
        return self.block(x)

# --------------------------
# 核心 Bottleneck
# --------------------------

class Bottleneck(nn.Module):
    def __init__(self, ins, outs, k, s, p, d, downsample=None):
        super().__init__()
        self.ds = downsample
        # 1x1 降维
        self.cbr1 = CBR(ins, outs, 1, 1, 0)
        # 3x3 处理 (注意这里的 dilation)
        self.cbr2 = CBR(outs, outs, 3, s, p, d)
        # 1x1 升维 (Expansion=4)
        self.cv = nn.Conv2d(outs, outs * 4, 1, 1, bias=False)
        self.bn = nn.BatchNorm2d(outs * 4)

    def forward(self, x):
        identity = x
        out = self.cbr1(x)
        out = self.cbr2(out)
        out = self.cv(out)
        out = self.bn(out)
        
        if self.ds is not None:
            identity = self.ds(x)
            
        out += identity
        return F.relu(out, inplace=True)

# --------------------------
# 各个 Layer 实现
# --------------------------

class Layer_1(nn.Module):
    def __init__(self, ins, outs, k=3, s=1, p=1, d=1):
        super().__init__()
        self.expansion = 4
        # Layer 1 输入通常与输出基数一致(64->64)，但Expansion后变256，所以需要Downsample
        ds = DownSample(ins, outs * self.expansion, 1, s)
        
        self.b1 = Bottleneck(ins, outs, k, s, p, d, downsample=ds)
        self.b2 = Bottleneck(outs * self.expansion, outs, k, 1, p, d)
        self.b3 = Bottleneck(outs * self.expansion, outs, k, 1, p, d)

    def forward(self, x):
        x = self.b1(x)
        x = self.b2(x)
        x = self.b3(x)
        return x

class Layer_2(nn.Module):
    def __init__(self, ins, outs): 
        # Ins: 256, Outs(Base): 128 -> Final: 512
        super().__init__()
        exp = 4
        self.b = nn.Sequential(
            # Stride=2 进行下采样
            Bottleneck(ins, outs, 3, 2, 1, 1, downsample=DownSample(ins, outs * exp, 1, 2)),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1),
            Bottleneck(outs * exp, outs, 3, 1, 1, 1)
        )
    def forward(self, x):
        return self.b(x)

class Layer_3(nn.Module):
    def __init__(self, ins, outs):
        # Ins: 512, Outs(Base): 256 -> Final: 1024
        super().__init__()
        exp = 4
        # 注意：这里你使用了 dilation=2 (p=2) 在后续 block，这是增加感受野的好方法
        self.b = nn.Sequential(
            # Stride=2 进行下采样
            Bottleneck(ins, outs, 3, 1, 1, 1, downsample=DownSample(ins, outs * exp, 1, 1)),
            Bottleneck(outs * exp, outs, 3, 1, 2, 2), # d=2, p=2
            Bottleneck(outs * exp, outs, 3, 1, 2, 2),
            Bottleneck(outs * exp, outs, 3, 1, 2, 2),
            Bottleneck(outs * exp, outs, 3, 1, 2, 2),
            Bottleneck(outs * exp, outs, 3, 1, 2, 2),
        )
    def forward(self, x):
        return self.b(x)

class Layer_4(nn.Module):
    def __init__(self, ins, outs):
        # Ins: 1024, Outs(Base): 512 -> Final: 2048
        super().__init__()
        exp = 4
        # 【关键点】FCN 常用技巧：Layer 4 不再进行下采样 (Stride=1)
        # 为了保持分辨率，使用空洞卷积 (Dilation=2 和 4)
        self.b = nn.Sequential(
            # Stride=1, Dilation=2 (Padding=2), Downsample Stride=1
            Bottleneck(ins, outs, 3, 1, 2, 2, downsample=DownSample(ins, outs * exp, 1, 1)),
            Bottleneck(outs * exp, outs, 3, 1, 4, 4), # Multi-grid 策略: d=4, p=4
            Bottleneck(outs * exp, outs, 3, 1, 4, 4),
        )
    def forward(self, x):
        return self.b(x)

# --------------------------
# FCN 主模型
# --------------------------

class FCN(nn.Module):
    def __init__(self, ins=3, num_classes=21):
        super().__init__()
        
        # 1. Stem (前处理)
        # 输入: [B, 3, 480, 480] -> 输出: [B, 64, 120, 120]
        # cbr(ins, outs, k, s, p, d=1):
        self.stem = nn.Sequential(
            CBR(ins, 64, 7, 2, 3),   # -> /2
            #k=3,s=2,p=1
            nn.MaxPool2d(3, 2, 1),   # -> /4
        )
        
        # 2. Backbone (骨干)
        # Layer 1: [B, 64, 120, 120] -> [B, 256, 120, 120] (无下采样)
        self.layer1 = Layer_1(ins=64, outs=64)
        
        # Layer 2: [B, 256, 120,120] -> [B, 512, 60, 60] (Stride=2)
        self.layer2 = Layer_2(ins=256, outs=128)
        
        # Layer 3: [B, 512, 60, 60] -> [B, 1024, 60, 60] 
        self.layer3 = Layer_3(ins=512, outs=256)
        
        # Layer 4: [B, 1024, 60, 60] -> [B, 2048, 60, 60] (Stride=1, Dilated)
        # 注意：这里输出尺寸依然是 60*60 因为 Layer4 没做 stride=2
        self.layer4 = Layer_4(ins=1024, outs=512)
        
        # 3. Head (分类头)
        # 输入必须是 Layer4 的输出通道 (2048)
        self.head = Head(in_channels=2048, num_classes=num_classes)

    def forward(self, x):
        input_size = x.shape[2:] # 记录 H, W: (224, 224)
        
        # --- 编码阶段 (Encoder) ---
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        # --- 解码/分类阶段 (Decoder/Head) ---
        # 此时 x 的形状应该是 [Batch, 2048, H/16, W/16]
        x = self.head(x) # -> [Batch, 21, H/16, W/16]
        
        # --- 上采样 (Upsampling) ---
        # 直接一次性双线性插值回到原图大小
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=True)
        
        return x

# --------------------------


In [26]:


dummy_input = torch.randn(2, 3, 480, 480)
model = FCN(ins=3, num_classes=21)

print("正在测试模型流向...")
try:
    output = model(dummy_input)
    print("\n✅ 模型运行成功!")
    print(f"输入尺寸: {dummy_input.shape}")
    print(f"输出尺寸: {output.shape}")
    
    # 验证 Layer 4 
    # 224 / 16 = 14
    stem_out = model.stem(dummy_input)
    l1_out = model.layer1(stem_out)
    l2_out = model.layer2(l1_out)
    l3_out = model.layer3(l2_out)
    l4_out = model.layer4(l3_out)
    
    print("\n--- 内部尺寸检查 ---")
    print(f"Layer 1 Output: {l1_out.shape} ")
    print(f"Layer 2 Output: {l2_out.shape} ")
    print(f"Layer 3 Output: {l3_out.shape} ")
    print(f"Layer 4 Output: {l4_out.shape} ")
    
except Exception as e:
    print("\n❌ 发生错误:")
    print(e)

正在测试模型流向...

✅ 模型运行成功!
输入尺寸: torch.Size([2, 3, 480, 480])
输出尺寸: torch.Size([2, 21, 480, 480])

--- 内部尺寸检查 ---
Layer 1 Output: torch.Size([2, 256, 120, 120]) 
Layer 2 Output: torch.Size([2, 512, 60, 60]) 
Layer 3 Output: torch.Size([2, 1024, 60, 60]) 
Layer 4 Output: torch.Size([2, 2048, 60, 60]) 
